In [1]:
%pip install langsmith

In [2]:
import os
from getpass import getpass
from langsmith import traceable

os.environ["LANGSMITH_API_KEY"] = getpass("API Key: ").strip()
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "MADAR_Project"

API Key: ··········


In [3]:
   from langsmith import Client
   try:
       next(iter(Client().list_projects()), None)
       print("✅ المفتاح شغال")
   except Exception as e:
       print("❌ المفتاح مرفوض:", e)

✅ المفتاح شغال


### **Satellite Agent**

In [4]:
class SatelliteAgent:
    def __init__(self, satellite_id, location, battery_level=100, visibility=None):
        self.satellite_id = satellite_id
        self.location = location
        self.battery_level = battery_level
        self.visibility = visibility or []
        self.availability = True
        self.assigned_mission = None
        self.satellite_status = "Available"
        self.mission_plan = []

    # Symbolic world model: current satellite state
    def worldModel(self):
        return {
            "location": self.location,
            "battery": self.battery_level,
            "coverage": self.visibility,
            "available": self.availability,
            "status": self.satellite_status
        }

    def checkAvailability(self):
        return self.availability

    @traceable(name="Satellite - Evaluate Suitability")
    def evaluateSuitability(self, mission):
        state = self.worldModel()
        return (state["available"] and
                state["status"] == "Available" and
                state["battery"] >= 30 and
                mission["target_location"] in state["coverage"])

    # Deliberative: create a plan based on mission requirements
    @traceable(name="Satellite - Plan Mission")
    def planMission(self, mission):
        if not self.evaluateSuitability(mission):
            return None

        state = self.worldModel()
        plan = ["Prepare satellite"]

        if state["location"] != mission["target_location"]:
            plan.append(f"Adjust observation to {mission['target_location']}")

        plan.append(f"Observe {mission['target_location']}")

        if mission.get("priority") == "High":
            plan.append("Prioritize data transmission")

        plan.append("Send mission results")
        return plan

    @traceable(name="Satellite - Assign Mission")
    def assignMission(self, mission):
        plan = self.planMission(mission)
        if plan is None:
            return False

        self.assigned_mission = mission
        self.mission_plan = plan
        self.availability = False
        self.satellite_status = "Busy"
        return True

    @traceable(name="Satellite - Execute Mission")
    def executeMission(self):
        if not self.assigned_mission:
            return "No mission assigned"
        if self.satellite_status in ["Low Battery", "Connection Lost"]:
            return f"Mission paused: {self.satellite_status}"
        if self.mission_plan:
            return self.mission_plan.pop(0)
        return self.completeMission()

    def completeMission(self):
        mission_id = self.assigned_mission["mission_id"]
        self.assigned_mission = None
        self.mission_plan = []
        self.availability = True
        self.satellite_status = "Available"
        return f"Mission {mission_id} completed"

    # Reactive: respond immediately to alerts
    @traceable(name="Satellite - Handle Alert")
    def handleAlert(self, alert):
        if alert in ["LOW_BATTERY", "CONNECTION_LOST"]:
            self.satellite_status = alert.replace("_", " ").title()
        return self.satellite_status

    def updateStatus(self):
        return {
            "satellite_id": self.satellite_id,
            "availability": self.availability,
            "battery_level": self.battery_level,
            "status": self.satellite_status
        }

### **Battery Agent**

In [5]:
class BatteryAgent:
    def __init__(self, battery_threshold=30):
        self.battery_threshold = battery_threshold
        self.battery_status = {}

    # Monitor the battery status of each satellite
    @traceable(name="Battery - Monitor")
    def monitorBattery(self, satellite):
        status = ("Low Battery" if satellite.battery_level < self.battery_threshold
                  else "Normal")
        self.battery_status[satellite.satellite_id] = status
        return status

    # Check if the battery level is sufficient
    @traceable(name="Battery - Check Level")
    def checkBatteryLevel(self, satellite):
        return satellite.battery_level >= self.battery_threshold

    # Reactive: send an alert when the battery is low
    @traceable(name="Battery - Low Battery Alert")
    def sendLowBatteryAlert(self, satellite):
        if self.checkBatteryLevel(satellite):
            return None

        satellite.handleAlert("LOW_BATTERY")
        return {
            "sender": "BatteryAgent",
            "receiver": "CoordinatorAgent",
            "satellite_id": satellite.satellite_id,
            "type": "LOW_BATTERY"
        }


# **Mission Agent**

In [6]:
class MissionAgent:
    def __init__(self):
        self.missions = {}

    @traceable(name="Mission - Receive Mission")
    def receiveMission(self, mission_id, target_location, priority, time_window):
        """
        Receives a new Earth observation mission
        and stores its requirements.
        """

        mission = {
            "mission_id": mission_id,
            "target_location": target_location,
            "priority": priority,
            "time_window": time_window,
            "mission_status": "Pending"
        }

        if not self.validateRequirements(mission):
            return None

        self.missions[mission_id] = mission
        return mission


    @traceable(name="Mission - Validate Requirements")
    def validateRequirements(self, mission):
        """
        Checks whether all required mission information is valid.
        """

        required_fields = [
            "mission_id",
            "target_location",
            "priority",
            "time_window"
        ]

        for field in required_fields:
            if not mission.get(field):
                return False

        valid_priorities = ["Low", "Medium", "High"]

        if mission["priority"] not in valid_priorities:
            return False

        return True


    @traceable(name="Mission - Update Status")
    def updateMissionStatus(self, mission_id, new_status):
        """
        Updates the current status of a mission.
        """

        if mission_id not in self.missions:
            return False

        valid_statuses = [
            "Pending",
            "Assigned",
            "In Progress",
            "Completed",
            "Failed",
            "Reassigned"
        ]

        if new_status not in valid_statuses:
            return False

        self.missions[mission_id]["mission_status"] = new_status
        return True


    def getMission(self, mission_id):
        return self.missions.get(mission_id)


    def getMissionRequirements(self, mission_id):
        mission = self.getMission(mission_id)

        if mission is None:
            return None

        return {
            "mission_id": mission["mission_id"],
            "target_location": mission["target_location"],
            "priority": mission["priority"],
            "time_window": mission["time_window"]
        }

# **Ground Station Agent**

In [7]:
class GroundStationAgent:
    def __init__(self, station_id="GS-1"):
        self.station_id = station_id
        self.station_status = "Active"
        self.mission_status = {}
        self.received_updates = []

    @traceable(name="Ground Station - Receive Mission Update")
    def receiveMissionUpdates(
        self,
        mission_id,
        satellite_id,
        status,
        details=None
    ):
        """
        Receives mission execution updates from a Satellite Agent.
        """

        update = {
            "mission_id": mission_id,
            "satellite_id": satellite_id,
            "status": status,
            "details": details
        }

        self.received_updates.append(update)
        self.mission_status[mission_id] = status

        return update


    @traceable(name="Ground Station - Monitor Mission Status")
    def monitorMissionStatus(self, mission_id):
        """
        Returns the latest known status of a mission.
        """

        return self.mission_status.get(
            mission_id,
            "No update available"
        )


    @traceable(name="Ground Station - Report Mission Status")
    def reportMissionStatus(self, mission_id):
        """
        Prepares a mission status report for the
        Mission Agent and Coordinator Agent.
        """

        status = self.monitorMissionStatus(mission_id)

        report = {
            "sender": "GroundStationAgent",
            "receivers": [
                "MissionAgent",
                "CoordinatorAgent"
            ],
            "mission_id": mission_id,
            "mission_status": status
        }

        return report

# **MADAR Simulation Environment**

In [8]:
class SimulationEnvironment:
    def __init__(self):

        # Satellites operating in the simulation
        self.satellites = {}

        # Earth observation missions
        self.missions = {}

        # Communication state of each satellite
        self.communication_status = {}

        # Discrete simulation step
        self.simulation_step = 0

        # Records important changes in the environment
        self.event_log = []


    def addSatellite(self, satellite):
        """
        Adds a Satellite Agent to the simulation.
        """

        self.satellites[satellite.satellite_id] = satellite

        self.communication_status[
            satellite.satellite_id
        ] = "Connected"

        self.event_log.append(
            f"{satellite.satellite_id} added to the environment"
        )


    def addMission(self, mission):
        """
        Adds a validated mission to the environment.
        """

        if mission is None:
            return False

        mission_id = mission["mission_id"]

        self.missions[mission_id] = mission

        self.event_log.append(
            f"Mission {mission_id} added to the environment"
        )

        return True


    def getSatellite(self, satellite_id):
        return self.satellites.get(satellite_id)


    def getMission(self, mission_id):
        return self.missions.get(mission_id)


    def getAvailableSatellites(self):
        """
        Returns satellites that are currently available.
        """

        return [
            satellite
            for satellite in self.satellites.values()
            if satellite.availability
        ]


    def updateCommunicationStatus(
        self,
        satellite_id,
        status
    ):
        """
        Updates communication state when the
        Communication Agent detects a change.
        """

        if satellite_id not in self.satellites:
            return False

        valid_statuses = [
            "Connected",
            "Disconnected"
        ]

        if status not in valid_statuses:
            return False

        self.communication_status[
            satellite_id
        ] = status

        self.event_log.append(
            f"{satellite_id} communication status changed to {status}"
        )

        return True


    def advanceSimulation(self):
        """
        Moves the environment to the next discrete step.
        """

        self.simulation_step += 1

        self.event_log.append(
            f"Simulation advanced to step {self.simulation_step}"
        )

        return self.simulation_step


    def getEnvironmentState(self):
        """
        Returns the current simulated system state.
        This can later be used by the main program or UI.
        """

        return {
            "simulation_step": self.simulation_step,

            "satellites": {
                satellite_id: satellite.updateStatus()
                for satellite_id, satellite
                in self.satellites.items()
            },

            "missions": self.missions,

            "communication_status":
                self.communication_status
        }


    def getEventLog(self):
        return self.event_log

# **Communication & Coordination**

In [9]:
class CommunicationAgent:
    VALID_PERFORMATIVES = {"REQUEST", "INFORM", "CFP", "PROPOSE", "REFUSE",
                           "ACCEPT_PROPOSAL", "REJECT_PROPOSAL"}

    def __init__(self, environment):
        self.environment = environment
        self.connection_status = {}
        self.communication_failure = []
        self.message_log = []
        self.message_counter = 0

    # ACL-like messaging: every message in the system is validated and logged here
    @traceable(name="Communication - Send Message")
    def sendMessage(self, initiator, responder, performative, content,
                    ontology="MADAR", expected_outcome=None):
        if performative not in self.VALID_PERFORMATIVES:
            raise ValueError("Invalid ACL performative")
        self.message_counter += 1
        message = {
            "message_id": self.message_counter, "initiator": initiator,
            "responder": responder, "performative": performative,
            "content": content, "ontology": ontology,
            "expected_outcome": expected_outcome
        }
        self.message_log.append(message)
        return message

    @traceable(name="Communication - Check Connection")
    def checkConnection(self, satellite_id):
        status = self.environment.communication_status.get(satellite_id, "Disconnected")
        self.connection_status[satellite_id] = status
        return status == "Connected"

    @traceable(name="Communication - Report Connection Loss")
    def reportConnectionLoss(self, satellite_id):
        if satellite_id not in self.communication_failure:
            self.communication_failure.append(satellite_id)
        satellite = self.environment.getSatellite(satellite_id)
        if satellite:
            satellite.handleAlert("CONNECTION_LOST")
        self.sendMessage("CommunicationAgent", "CoordinatorAgent", "INFORM",
                         {"event": "CONNECTION_LOST", "satellite_id": satellite_id},
                         expected_outcome="Coordinator handles the unavailable satellite")
        return {"satellite_id": satellite_id, "type": "CONNECTION_LOST"}

    @traceable(name="Communication - Monitor Communication")
    def monitorCommunication(self):
        alerts = []
        for satellite_id in self.environment.satellites:
            if not self.checkConnection(satellite_id) and satellite_id not in self.communication_failure:
                alerts.append(self.reportConnectionLoss(satellite_id))
        return alerts

    # Used to change a satellite connection (e.g. from the UI); the loss itself
    # is detected and reported later by monitorCommunication()
    @traceable(name="Communication - Update Connection Status")
    def updateConnectionStatus(self, satellite_id, status):
        if not self.environment.updateCommunicationStatus(satellite_id, status):
            return False
        self.connection_status[satellite_id] = status
        if status == "Connected" and satellite_id in self.communication_failure:
            self.communication_failure.remove(satellite_id)  # allow future losses to be reported
        return True

    def getMessageLog(self):
        return self.message_log


class CoordinatorAgent:
    ALERT_TYPES = ("LOW_BATTERY", "CONNECTION_LOST", "EXECUTION_FAILED")

    def __init__(self, environment, mission_agent, communication_agent,
                 battery_agent=None, ground_station_agent=None, battery_threshold=30):
        self.environment = environment
        self.mission_agent = mission_agent
        self.communication_agent = communication_agent
        self.battery_agent = battery_agent
        self.ground_station_agent = ground_station_agent
        self.battery_threshold = battery_threshold
        self.mission_queue = []
        self.satellite_status = {}
        self.mission_priority = {}
        self.assignment_status = {}
        self.available_satellites = []
        self.scheduling_conflicts = []
        self.successful_reassignments = []

    @traceable(name="Coordinator - Receive Mission")
    def receiveMission(self, mission_id):
        mission = self.mission_agent.getMission(mission_id)
        if mission is None:
            return False
        if mission_id not in self.mission_queue:
            self.mission_queue.append(mission_id)
        self.mission_priority[mission_id] = mission["priority"]
        self.assignment_status[mission_id] = "Pending"
        self.communication_agent.sendMessage("MissionAgent", "CoordinatorAgent", "INFORM",
                                             {"mission_id": mission_id, "mission_details": dict(mission)},
                                             expected_outcome="Coordinator receives the mission request")
        return True

    # Contract-Net style round: CFP to every satellite, then PROPOSE or REFUSE back
    # (the Coordinator answers proposals with ACCEPT_PROPOSAL / REJECT_PROPOSAL)
    @traceable(name="Coordinator - Request Evaluation (CFP)")
    def requestEvaluation(self, mission_id):
        mission = self.mission_agent.getMission(mission_id)
        if mission is None:
            return []
        self.communication_agent.sendMessage("CoordinatorAgent", "MissionAgent", "REQUEST",
                                             {"mission_id": mission_id},
                                             expected_outcome="Mission Agent provides mission requirements")
        proposals = []
        for satellite_id, satellite in self.environment.satellites.items():
            content = {"mission_id": mission_id, "target_location": mission["target_location"],
                       "priority": mission["priority"], "time_window": mission["time_window"]}
            self.communication_agent.sendMessage("CoordinatorAgent", f"SatelliteAgent-{satellite_id}", "CFP",
                                                 content, expected_outcome="Satellite evaluates mission suitability")
            suitable = (self.communication_agent.checkConnection(satellite_id)
                        and satellite.battery_level >= self.battery_threshold
                        and satellite.evaluateSuitability(mission))
            if suitable:
                proposal = {"satellite_id": satellite_id, "battery_level": satellite.battery_level,
                            "mission_id": mission_id}
                proposals.append(proposal)
                self.communication_agent.sendMessage(f"SatelliteAgent-{satellite_id}", "CoordinatorAgent", "PROPOSE",
                                                     proposal, expected_outcome="Coordinator considers satellite proposal")
            else:
                self.communication_agent.sendMessage(f"SatelliteAgent-{satellite_id}", "CoordinatorAgent", "REFUSE",
                                                     {"satellite_id": satellite_id, "mission_id": mission_id,
                                                      "reason": "Unavailable or unsuitable"},
                                                     expected_outcome="Coordinator rejects this satellite")
        return proposals

    # The most suitable satellite = the proposal with the highest battery level
    @traceable(name="Coordinator - Select Satellite")
    def selectSatellite(self, proposals):
        if not proposals:
            return None
        best = max(proposals, key=lambda proposal: proposal["battery_level"])
        return self.environment.getSatellite(best["satellite_id"])

    # Scheduling conflict: the mission needs a satellite that is already busy
    @traceable(name="Coordinator - Detect Conflict")
    def detectConflict(self, mission_id, satellite):
        if satellite is None:
            return False
        if satellite.assigned_mission is not None or not satellite.availability:
            conflict = {"mission_id": mission_id, "satellite_id": satellite.satellite_id,
                        "reason": "Satellite is already assigned to another mission"}
            if conflict not in self.scheduling_conflicts:
                self.scheduling_conflicts.append(conflict)
            return True
        return False

    @traceable(name="Coordinator - Assign Mission")
    def assignMission(self, mission_id, satellite=None):
        mission = self.mission_agent.getMission(mission_id)
        if mission is None:
            return False
        proposals = []
        if satellite is None:
            proposals = self.requestEvaluation(mission_id)
            satellite = self.selectSatellite(proposals)
            if satellite is None:
                # No free satellite: it is a scheduling conflict if a satellite
                # that covers the target is busy with another mission
                for busy in self.environment.satellites.values():
                    if busy.assigned_mission and mission["target_location"] in busy.visibility:
                        self.detectConflict(mission_id, busy)
                        break
                self.assignment_status[mission_id] = "Pending"
                return False
        if (self.detectConflict(mission_id, satellite)
                or not self.communication_agent.checkConnection(satellite.satellite_id)
                or satellite.battery_level < self.battery_threshold):
            return False
        if not satellite.assignMission(mission):
            return False
        self.assignment_status[mission_id] = "Assigned"
        self.mission_agent.updateMissionStatus(mission_id, "Assigned")
        self.communication_agent.sendMessage("CoordinatorAgent", f"SatelliteAgent-{satellite.satellite_id}", "ACCEPT_PROPOSAL",
                                             {"mission_id": mission_id, "satellite_id": satellite.satellite_id},
                                             expected_outcome="Satellite accepts the mission assignment")
        # Contract-Net: tell every other proposer that its proposal was not selected
        for proposal in proposals:
            if proposal["satellite_id"] != satellite.satellite_id:
                self.communication_agent.sendMessage("CoordinatorAgent", f"SatelliteAgent-{proposal['satellite_id']}", "REJECT_PROPOSAL",
                                                     {"mission_id": mission_id, "satellite_id": proposal["satellite_id"],
                                                      "reason": "Another satellite was selected (higher battery)"},
                                                     expected_outcome="Satellite stays available for other missions")
        self._notifyGroundStation(mission_id, satellite.satellite_id, "Assigned",
                                  "Mission assigned successfully")
        return True

    # Distribute waiting missions: High priority first, then Medium, then Low
    @traceable(name="Coordinator - Assign Pending Missions")
    def assignPendingMissions(self):
        rank = {"High": 0, "Medium": 1, "Low": 2}
        pending = [m for m in self.mission_queue if self.assignment_status.get(m) == "Pending"]
        pending.sort(key=lambda m: rank[self.mission_priority[m]])
        return {mission_id: self.assignMission(mission_id) for mission_id in pending}

    @traceable(name="Coordinator - Handle Alert")
    def handleAlert(self, alert):
        if not isinstance(alert, dict):
            return False
        satellite_id, alert_type = alert.get("satellite_id"), alert.get("type")
        satellite = self.environment.getSatellite(satellite_id)
        if satellite is None or alert_type not in self.ALERT_TYPES:
            return False
        if alert_type == "EXECUTION_FAILED":
            satellite.satellite_status = "Execution Failed"
            self.communication_agent.sendMessage(f"SatelliteAgent-{satellite_id}", "CoordinatorAgent", "INFORM", alert,
                                                 expected_outcome="Coordinator responds to satellite failure")
        else:
            satellite.handleAlert(alert_type)
            if alert_type == "LOW_BATTERY":
                self.communication_agent.sendMessage("BatteryAgent", "CoordinatorAgent", "INFORM", alert,
                                                     expected_outcome="Coordinator responds to satellite failure")
        return self.reassignMission(satellite_id)

    # Move the mission of a failed satellite to another suitable satellite
    @traceable(name="Coordinator - Reassign Mission")
    def reassignMission(self, satellite_id):
        failed = self.environment.getSatellite(satellite_id)
        if failed is None or failed.assigned_mission is None:
            return False
        mission_id = failed.assigned_mission["mission_id"]
        failed.assigned_mission = None
        failed.availability = False
        self.assignment_status[mission_id] = "Reassigning"
        self.communication_agent.sendMessage("CoordinatorAgent", "MissionAgent", "INFORM",
                                             {"mission_id": mission_id, "event": "MISSION_REASSIGNMENT_REQUIRED",
                                              "failed_satellite": satellite_id},
                                             expected_outcome="Mission Agent is informed about reassignment")
        if self.assignMission(mission_id):
            self.mission_agent.updateMissionStatus(mission_id, "Reassigned")
            self.assignment_status[mission_id] = "Reassigned"
            self.successful_reassignments.append(mission_id)
            self._notifyGroundStation(mission_id, self._getAssignedSatelliteID(mission_id),
                                      "Reassigned", "Mission successfully reassigned")
            return True
        self.assignment_status[mission_id] = "Pending"
        self.mission_agent.updateMissionStatus(mission_id, "Pending")
        return False

    # INFORM message to the Ground Station + update of its records
    def _notifyGroundStation(self, mission_id, satellite_id, status, details):
        if not self.ground_station_agent:
            return
        self.communication_agent.sendMessage(f"SatelliteAgent-{satellite_id}", "GroundStationAgent", "INFORM",
                                             {"mission_id": mission_id, "status": status},
                                             expected_outcome="Ground Station records the mission status")
        self.ground_station_agent.receiveMissionUpdates(mission_id, satellite_id, status, details)

    def _getAssignedSatelliteID(self, mission_id):
        for satellite_id, satellite in self.environment.satellites.items():
            if satellite.assigned_mission and satellite.assigned_mission.get("mission_id") == mission_id:
                return satellite_id
        return None

    # Collect battery and communication alerts and handle each one
    @traceable(name="Coordinator - Process Alerts")
    def processAlerts(self):
        alerts = []
        if self.battery_agent:
            for satellite in self.environment.satellites.values():
                self.battery_agent.monitorBattery(satellite)
                if satellite.satellite_status == "Low Battery":
                    continue  # already reported, do not repeat the same alert every step
                alert = self.battery_agent.sendLowBatteryAlert(satellite)
                if alert:
                    alerts.append(alert)
        alerts.extend(self.communication_agent.monitorCommunication())
        return [self.handleAlert(alert) for alert in alerts]

    def updateSatelliteStatus(self):
        self.satellite_status, self.available_satellites = {}, []
        for satellite_id, satellite in self.environment.satellites.items():
            self.satellite_status[satellite_id] = satellite.satellite_status
            if (satellite.availability and satellite.battery_level >= self.battery_threshold
                    and self.communication_agent.checkConnection(satellite_id)):
                self.available_satellites.append(satellite_id)
        return self.satellite_status

    def getCoordinatorState(self):
        self.updateSatelliteStatus()
        return {"mission_queue": self.mission_queue, "satellite_status": self.satellite_status,
                "mission_priority": self.mission_priority, "assignment_status": self.assignment_status,
                "available_satellites": self.available_satellites,
                "scheduling_conflicts": self.scheduling_conflicts,
                "successful_reassignments": self.successful_reassignments}

# **Agent Lifecycle (Perception → Reasoning → Action)**


In [10]:
environment = SimulationEnvironment()

print("Environment created successfully")

Environment created successfully


In [11]:
satellite1 = SatelliteAgent(
    satellite_id="SAT-1",
    location="Riyadh",
    battery_level=90,
    visibility=["Riyadh", "Jeddah"]
)

satellite2 = SatelliteAgent(
    satellite_id="SAT-2",
    location="Jeddah",
    battery_level=80,
    visibility=["Riyadh", "Jeddah"]
)

environment.addSatellite(satellite1)
environment.addSatellite(satellite2)

print("Satellites added successfully")

Satellites added successfully


In [12]:
mission_agent = MissionAgent()

battery_agent = BatteryAgent()

ground_station_agent = GroundStationAgent()

In [13]:
communication_agent = CommunicationAgent(environment)

coordinator_agent = CoordinatorAgent(
    environment=environment,
    mission_agent=mission_agent,
    communication_agent=communication_agent,
    battery_agent=battery_agent,
    ground_station_agent=ground_station_agent
)

print("All agents created successfully")

All agents created successfully


In [14]:
mission = mission_agent.receiveMission(
    mission_id="M001",
    target_location="Riyadh",
    priority="High",
    time_window="10:00-12:00"
)

print(mission)

{'mission_id': 'M001', 'target_location': 'Riyadh', 'priority': 'High', 'time_window': '10:00-12:00', 'mission_status': 'Pending'}


### **Test: Communication & Coordination**


In [15]:
def new_system():
    env = SimulationEnvironment()
    env.addSatellite(SatelliteAgent("SAT-1", "Riyadh", 90, ["Riyadh", "Jeddah"]))
    env.addSatellite(SatelliteAgent("SAT-2", "Jeddah", 80, ["Riyadh", "Jeddah"]))
    ma, ba, gs = MissionAgent(), BatteryAgent(), GroundStationAgent()
    ca = CommunicationAgent(env)
    return env, ma, ca, CoordinatorAgent(env, ma, ca, ba, gs)

def add_mission(ma, co, mission_id, priority="High"):
    ma.receiveMission(mission_id, "Riyadh", priority, "10:00-12:00")
    co.receiveMission(mission_id)

def print_log(ca):
    for m in ca.getMessageLog():
        print(f"{m['message_id']:>2}. {m['initiator']} -> {m['responder']} [{m['performative']}]")

def owner(env, mission_id):
    return [s for s, sat in env.satellites.items()
            if sat.assigned_mission and sat.assigned_mission["mission_id"] == mission_id]

# 1) Normal assignment: CFP -> PROPOSE -> ACCEPT, best battery wins
env, ma, ca, co = new_system()
add_mission(ma, co, "M001")
co.assignMission("M001")
print("TEST 1 - Normal assignment ->", owner(env, "M001"))
print_log(ca)

# 2) Low battery: the mission is reassigned
env, ma, ca, co = new_system()
add_mission(ma, co, "M001"); co.assignMission("M001")
env.getSatellite("SAT-1").battery_level = 20
co.processAlerts()
print("\nTEST 2 - Low battery ->", owner(env, "M001"), co.successful_reassignments)

# 3) Communication failure: the mission is reassigned
env, ma, ca, co = new_system()
add_mission(ma, co, "M001"); co.assignMission("M001")
ca.updateConnectionStatus("SAT-1", "Disconnected")
co.processAlerts()
print("TEST 3 - Communication failure ->", owner(env, "M001"), co.successful_reassignments)

# 4) Scheduling conflict: 3 missions, 2 satellites -> highest priorities first, the last one waits
env, ma, ca, co = new_system()
add_mission(ma, co, "M1", "Low"); add_mission(ma, co, "M2", "High"); add_mission(ma, co, "M3", "Medium")
print("TEST 4 - Scheduling conflict ->", co.assignPendingMissions())
print("   conflicts:", co.scheduling_conflicts)
sat = env.getSatellite("SAT-1")
while sat.assigned_mission:
    sat.executeMission()          # SAT-1 finishes its mission
print("   after SAT-1 is free ->", co.assignPendingMissions())

# 5) Execution failure: the mission is reassigned
env, ma, ca, co = new_system()
add_mission(ma, co, "M001"); co.assignMission("M001")
co.handleAlert({"satellite_id": "SAT-1", "type": "EXECUTION_FAILED"})
print("TEST 5 - Execution failure ->", owner(env, "M001"), co.successful_reassignments)

TEST 1 - Normal assignment -> ['SAT-1']
 1. MissionAgent -> CoordinatorAgent [INFORM]
 2. CoordinatorAgent -> MissionAgent [REQUEST]
 3. CoordinatorAgent -> SatelliteAgent-SAT-1 [CFP]
 4. SatelliteAgent-SAT-1 -> CoordinatorAgent [PROPOSE]
 5. CoordinatorAgent -> SatelliteAgent-SAT-2 [CFP]
 6. SatelliteAgent-SAT-2 -> CoordinatorAgent [PROPOSE]
 7. CoordinatorAgent -> SatelliteAgent-SAT-1 [ACCEPT_PROPOSAL]
 8. CoordinatorAgent -> SatelliteAgent-SAT-2 [REJECT_PROPOSAL]
 9. SatelliteAgent-SAT-1 -> GroundStationAgent [INFORM]

TEST 2 - Low battery -> ['SAT-2'] ['M001']
TEST 3 - Communication failure -> ['SAT-2'] ['M001']
TEST 4 - Scheduling conflict -> {'M2': True, 'M3': True, 'M1': False}
   conflicts: [{'mission_id': 'M1', 'satellite_id': 'SAT-1', 'reason': 'Satellite is already assigned to another mission'}]
   after SAT-1 is free -> {'M1': True}
TEST 5 - Execution failure -> ['SAT-2'] ['M001']


# **Testing Scenarios, Main Execution & User Interface**

In [16]:
import json
import time
from copy import deepcopy

import pandas as pd
from langsmith import traceable

SCENARIOS = (
    "Normal Mission Assignment",
    "Low Battery",
    "Communication Failure",
    "Scheduling Conflict",
)


def _create_phase4_system(sat1_battery=90, sat2_battery=80, battery_threshold=30):
    """Create a fresh simulation so consecutive UI runs cannot contaminate one another."""
    env = SimulationEnvironment()
    env.addSatellite(SatelliteAgent("SAT-1", "Riyadh", int(sat1_battery), ["Riyadh", "Jeddah"]))
    env.addSatellite(SatelliteAgent("SAT-2", "Jeddah", int(sat2_battery), ["Riyadh", "Jeddah"]))
    mission_agent = MissionAgent()
    battery_agent = BatteryAgent(battery_threshold=int(battery_threshold))
    ground_station = GroundStationAgent()
    communication_agent = CommunicationAgent(env)
    coordinator = CoordinatorAgent(
        environment=env,
        mission_agent=mission_agent,
        communication_agent=communication_agent,
        battery_agent=battery_agent,
        ground_station_agent=ground_station,
        battery_threshold=int(battery_threshold),
    )
    return env, mission_agent, battery_agent, ground_station, communication_agent, coordinator


def _add_phase4_mission(env, mission_agent, coordinator, mission_id, priority):
    mission = mission_agent.receiveMission(mission_id, "Riyadh", priority, "10:00-12:00")
    if mission is None or not env.addMission(mission):
        raise ValueError(f"Mission {mission_id} could not be created")
    if not coordinator.receiveMission(mission_id):
        raise ValueError(f"Coordinator rejected {mission_id}")


def _phase4_metrics(env, mission_agent, coordinator):
    total = len(env.missions)
    completed = sum(
        mission["mission_status"] == "Completed"
        for mission in mission_agent.missions.values()
    )
    return {
        "total_missions": total,
        "completed_missions": completed,
        "mission_completion_rate_pct": round(100 * completed / total, 2) if total else 0.0,
        "scheduling_conflicts": len(coordinator.scheduling_conflicts),
        "successful_mission_reassignments": len(set(coordinator.successful_reassignments)),
    }


def _phase4_snapshot(stage, env, mission_agent, battery_agent,
                     ground_station, communication_agent, coordinator):
    """Copy each step's status, not live object references."""
    agent_rows = []
    for satellite_id, satellite in env.satellites.items():
        mission_id = (
            satellite.assigned_mission["mission_id"]
            if satellite.assigned_mission else "—"
        )
        agent_rows.append({
            "Agent": "Satellite Agent",
            "ID": satellite_id,
            "Status": satellite.satellite_status,
            "Details": (f"Battery {satellite.battery_level}% | "
                        f"Link {env.communication_status[satellite_id]} | "
                        f"Mission {mission_id}"),
        })
    low_count = sum(
        s.battery_level < battery_agent.battery_threshold
        for s in env.satellites.values()
    )
    disconnected_count = sum(
        state == "Disconnected" for state in env.communication_status.values()
    )
    pending_count = sum(
        m["mission_status"] == "Pending" for m in mission_agent.missions.values()
    )
    agent_rows.extend([
        {
            "Agent": "Mission Agent", "ID": "MISSION",
            "Status": "Active", "Details": f"{len(mission_agent.missions)} requests; {pending_count} pending",
        },
        {
            "Agent": "Battery Agent", "ID": "BATTERY",
            "Status": "Low battery detected" if low_count else "Monitoring",
            "Details": f"Threshold {battery_agent.battery_threshold}% | Low {low_count}",
        },
        {
            "Agent": "Communication Agent", "ID": "COMMS",
            "Status": "Link failure" if disconnected_count else "Monitoring",
            "Details": f"Disconnected satellites {disconnected_count}",
        },
        {
            "Agent": "Coordinator Agent", "ID": "COORD",
            "Status": "Active",
            "Details": (f"Conflicts {len(coordinator.scheduling_conflicts)} | "
                        f"Reassignments {len(set(coordinator.successful_reassignments))}"),
        },
        {
            "Agent": "Ground Station Agent", "ID": ground_station.station_id,
            "Status": ground_station.station_status,
            "Details": f"Received updates {len(ground_station.received_updates)}",
        },
    ])

    mission_rows = []
    for mission_id, mission in mission_agent.missions.items():
        mission_rows.append({
            "Mission": mission_id,
            "Priority": mission["priority"],
            "Status": mission["mission_status"],
            "Satellite": coordinator._getAssignedSatelliteID(mission_id) or "—",
        })

    message_rows = []
    for msg in communication_agent.getMessageLog():
        message_rows.append({
            "ID": msg["message_id"],
            "From": msg["initiator"],
            "To": msg["responder"],
            "Performative": msg["performative"],
            "Content": json.dumps(msg["content"], ensure_ascii=False, default=str),
        })

    return deepcopy({
        "step": env.simulation_step,
        "stage": stage,
        "agent_rows": agent_rows,
        "mission_rows": mission_rows,
        "message_rows": message_rows,
        "metrics": _phase4_metrics(env, mission_agent, coordinator),
        "event_log": list(env.event_log),
    })


@traceable(name="MADAR - Main Scenario Execution", run_type="chain")
def run_madar_scenario(
    scenario="Normal Mission Assignment",
    sat1_battery=90,
    sat2_battery=80,
    battery_threshold=30,
    conflict_missions=3,
):
    """Orchestrate EXISTING agent code, test one scenario, and collect UI snapshots.

    Scenario inputs and final outputs are traced as one parent LangSmith run.
    Nested agent traces belong to existing team members' implementations.
    """
    if scenario not in SCENARIOS:
        raise ValueError(f"Unknown scenario: {scenario}")
    sat1_battery, sat2_battery, battery_threshold = (
        int(sat1_battery), int(sat2_battery), int(battery_threshold)
    )
    conflict_missions = int(conflict_missions)
    if not (0 <= sat1_battery <= 100 and 0 <= sat2_battery <= 100):
        raise ValueError("Satellite batteries must be between 0 and 100")
    # Satellite Agent already enforces a built-in minimum of 30%.
    if not (30 <= battery_threshold <= 80):
        raise ValueError("Battery threshold must be in 30..80%")
    if not (3 <= conflict_missions <= 5):
        raise ValueError("Conflict test mission count must be in 3..5")

    env, ma, ba, gs, ca, co = _create_phase4_system(
        sat1_battery, sat2_battery, battery_threshold
    )
    history = []

    def capture(stage):
        env.event_log.append(f"[Step {env.simulation_step}] {stage}")
        history.append(_phase4_snapshot(stage, env, ma, ba, gs, ca, co))

    if scenario == "Scheduling Conflict":
        initial = [("M001", "Low"), ("M002", "High"), ("M003", "Medium")]
        extra = [(f"M{i:03}", "Low") for i in range(4, conflict_missions + 1)]
        for mission_id, priority in initial + extra:
            _add_phase4_mission(env, ma, co, mission_id, priority)
    else:
        _add_phase4_mission(env, ma, co, "M001", "High")

    capture("Mission requests received")
    co.assignPendingMissions()
    initial_owner = co._getAssignedSatelliteID("M001")
    initially_assigned = {
        mid: co._getAssignedSatelliteID(mid) for mid in ma.missions
    }
    initial_statuses = {
        mid: ma.getMission(mid)["mission_status"] for mid in ma.missions
    }
    capture("Initial satellite allocation completed")

    replacement_owner = None
    if scenario in {"Low Battery", "Communication Failure"} and initial_owner:
        if scenario == "Low Battery":
            sat = env.getSatellite(initial_owner)
            sat.battery_level = battery_threshold - 1
            capture(f"Injected low battery into {initial_owner}")
        else:
            ca.updateConnectionStatus(initial_owner, "Disconnected")
            capture(f"Disconnected {initial_owner}")
        co.processAlerts()  # Existing Battery / Communication / Coordinator behavior.
        replacement_owner = co._getAssignedSatelliteID("M001")
        capture("Failure alert processed; attempted mission reassignment")

    # Run until every possible mission is completed or the simulation is blocked.
    for _ in range(100):
        active = [s for s in env.satellites.values() if s.assigned_mission]
        pending = [m for m in ma.missions.values() if m["mission_status"] == "Pending"]
        if not active and not pending:
            break
        if not active:
            co.assignPendingMissions()
            active = [s for s in env.satellites.values() if s.assigned_mission]
            if not active:
                capture("No eligible satellite: remaining missions stay pending")
                break

        for satellite in active:
            mission_id = satellite.assigned_mission["mission_id"]
            if ma.getMission(mission_id)["mission_status"] != "In Progress":
                ma.updateMissionStatus(mission_id, "In Progress")
                co.assignment_status[mission_id] = "In Progress"
                co._notifyGroundStation(mission_id, satellite.satellite_id,
                                        "In Progress", "Mission execution started")
            action = satellite.executeMission()  # Existing satellite execution logic.
            env.advanceSimulation()
            env.event_log.append(
                f"{satellite.satellite_id} | {mission_id} | {action}"
            )
            if satellite.assigned_mission is None and action == f"Mission {mission_id} completed":
                ma.updateMissionStatus(mission_id, "Completed")
                co.assignment_status[mission_id] = "Completed"
                co._notifyGroundStation(mission_id, satellite.satellite_id,
                                        "Completed", "Mission executed successfully")
            capture(f"{satellite.satellite_id}: {action}")

        if any(m["mission_status"] == "Pending" for m in ma.missions.values()):
            if any(s.availability for s in env.satellites.values()):
                co.assignPendingMissions()
                capture("Attempted allocation of waiting missions")
    else:
        capture("Safety stop: maximum simulation iterations reached")

    metrics = _phase4_metrics(env, ma, co)
    protocol = {msg["performative"] for msg in ca.getMessageLog()}
    completed_all = metrics["completed_missions"] == metrics["total_missions"]
    if scenario == "Normal Mission Assignment":
        passed = bool(completed_all and metrics["total_missions"] == 1
                      and {"CFP", "PROPOSE", "ACCEPT_PROPOSAL"}.issubset(protocol))
    elif scenario == "Low Battery":
        passed = bool(completed_all and initial_owner and replacement_owner
                      and initial_owner != replacement_owner
                      and metrics["successful_mission_reassignments"] == 1)
    elif scenario == "Communication Failure":
        passed = bool(completed_all and initial_owner and replacement_owner
                      and initial_owner != replacement_owner
                      and metrics["successful_mission_reassignments"] == 1)
    else:
        passed = bool(completed_all and metrics["scheduling_conflicts"] >= 1
                      and initial_statuses.get("M002") == "Assigned"
                      and initial_statuses.get("M003") == "Assigned"
                      and initial_statuses.get("M001") == "Pending")
    capture("FINAL RESULT: " + ("PASS" if passed else "CHECK CONFIGURATION / FAIL"))

    return {
        "scenario": scenario,
        "inputs": {
            "SAT-1 battery": sat1_battery,
            "SAT-2 battery": sat2_battery,
            "battery threshold": battery_threshold,
            "conflict mission count": conflict_missions,
        },
        "passed": passed,
        "metrics": metrics,
        "initial_owner": initial_owner,
        "replacement_owner": replacement_owner,
        "initial_assignments": initially_assigned,
        "initial_statuses": initial_statuses,
        "final_mission_statuses": {
            mid: mission["mission_status"] for mid, mission in ma.missions.items()
        },
        "ground_station_updates": deepcopy(gs.received_updates),
        "scheduling_conflicts": deepcopy(co.scheduling_conflicts),
        "successful_reassignments": list(co.successful_reassignments),
        "message_log": deepcopy(ca.getMessageLog()),
        "history": history,
    }


def verify_madar_phase4():
    """Automated tests using default conditions; each case gets fresh agents."""
    rows = []
    for scenario in SCENARIOS:
        result = run_madar_scenario(scenario)
        metrics = result["metrics"]
        rows.append({
            "Scenario": scenario,
            "Test": "PASS" if result["passed"] else "FAIL",
            "Completed": f"{metrics['completed_missions']}/{metrics['total_missions']}",
            "Completion rate (%)": metrics["mission_completion_rate_pct"],
            "Scheduling conflicts": metrics["scheduling_conflicts"],
            "Successful reassignments": metrics["successful_mission_reassignments"],
        })
    return pd.DataFrame(rows)




In [17]:
verification_table = verify_madar_phase4()
display(verification_table)
assert (verification_table["Test"] == "PASS").all(), "Review failed scenarios before reporting results"


,Scenario,Test,Completed,Completion rate (%),Scheduling conflicts,Successful reassignments
0,Normal Mission Assignment,PASS,1/1,100.0,0,0
1,Low Battery,PASS,1/1,100.0,0,1
2,Communication Failure,PASS,1/1,100.0,0,1
3,Scheduling Conflict,PASS,3/3,100.0,1,0


# **LangSmith Evaluation & Dataset Management**

In [18]:
from langsmith import Client, evaluate

ls_client = Client()
DATASET_NAME = "MADAR - Scenario Test Set"

# Dataset-based testing: defined inputs + expected outputs for each scenario
TEST_CASES = [
    ({"scenario": "Normal Mission Assignment"},
     {"passed": True, "completion_rate": 100.0, "reassignments": 0, "conflicts": 0,
      "initial_owner": "SAT-1", "replacement_owner": None}),
    ({"scenario": "Low Battery"},
     {"passed": True, "completion_rate": 100.0, "reassignments": 1, "conflicts": 0,
      "initial_owner": "SAT-1", "replacement_owner": "SAT-2"}),
    ({"scenario": "Communication Failure"},
     {"passed": True, "completion_rate": 100.0, "reassignments": 1, "conflicts": 0,
      "initial_owner": "SAT-1", "replacement_owner": "SAT-2"}),
    ({"scenario": "Scheduling Conflict", "conflict_missions": 3},
     {"passed": True, "completion_rate": 100.0, "reassignments": 0, "conflicts": 1,
      "initial_owner": None, "replacement_owner": None}),
    ({"scenario": "Scheduling Conflict", "conflict_missions": 5},
     {"passed": True, "completion_rate": 100.0, "reassignments": 0, "conflicts": 3,
      "initial_owner": None, "replacement_owner": None}),
    # Edge case: the backup satellite is too weak, so the mission correctly cannot be reassigned
    ({"scenario": "Low Battery", "sat2_battery": 20},
     {"passed": False, "completion_rate": 0.0, "reassignments": 0, "conflicts": 0,
      "initial_owner": "SAT-1", "replacement_owner": None}),
    # Edge case: SAT-2 has the higher battery, so it must win the Contract-Net round
    ({"scenario": "Communication Failure", "sat1_battery": 50, "sat2_battery": 95},
     {"passed": True, "completion_rate": 100.0, "reassignments": 1, "conflicts": 0,
      "initial_owner": "SAT-2", "replacement_owner": "SAT-1"}),
]

# Creation & import (only the first time)
if ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
    print("Using existing dataset:", DATASET_NAME)
else:
    dataset = ls_client.create_dataset(
        DATASET_NAME,
        description="MADAR multi-agent satellite scenarios with expected outcomes",
    )
    for inputs, expected in TEST_CASES:
        ls_client.create_example(inputs=inputs, outputs=expected, dataset_id=dataset.id)
    print(f"Created dataset '{DATASET_NAME}' with {len(TEST_CASES)} examples")


# Target: runs the real multi-agent system for one dataset example
def madar_target(inputs: dict) -> dict:
    result = run_madar_scenario(**inputs)
    metrics = result["metrics"]
    return {
        "passed": result["passed"],
        "completion_rate": metrics["mission_completion_rate_pct"],
        "reassignments": metrics["successful_mission_reassignments"],
        "conflicts": metrics["scheduling_conflicts"],
        "initial_owner": result["initial_owner"],
        "replacement_owner": result["replacement_owner"],
        "protocol": sorted({m["performative"] for m in result["message_log"]}),
    }


# Custom evaluators (each returns a 0/1 score that appears as feedback in LangSmith)
def completion_rate_match(outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "completion_rate_match",
            "score": int(outputs["completion_rate"] == reference_outputs["completion_rate"])}


def correct_satellite_selected(outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "correct_satellite_selected",
            "score": int(outputs["initial_owner"] == reference_outputs["initial_owner"])}


def reassignment_correct(outputs: dict, reference_outputs: dict) -> dict:
    ok = (outputs["reassignments"] == reference_outputs["reassignments"]
          and outputs["replacement_owner"] == reference_outputs["replacement_owner"])
    return {"key": "reassignment_correct", "score": int(ok)}


def conflict_detection_correct(outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "conflict_detection_correct",
            "score": int(outputs["conflicts"] == reference_outputs["conflicts"])}


def contract_net_protocol(outputs: dict) -> dict:
    required = {"CFP", "PROPOSE", "ACCEPT_PROPOSAL"}
    return {"key": "contract_net_protocol",
            "score": int(required.issubset(outputs["protocol"]))}


def scenario_verdict_match(outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "scenario_verdict_match",
            "score": int(outputs["passed"] == reference_outputs["passed"])}


MADAR_EVALUATORS = [completion_rate_match, correct_satellite_selected, reassignment_correct,
                    conflict_detection_correct, contract_net_protocol, scenario_verdict_match]

# Each run of this cell = one Experiment in LangSmith (Run1, Run2, ... comparable side by side)
experiment = evaluate(
    madar_target,
    data=DATASET_NAME,
    evaluators=MADAR_EVALUATORS,
    experiment_prefix="MADAR-eval",
    description="Rule-based multi-agent coordination evaluated against expected scenario outcomes",
    max_concurrency=1,
)

try:
    df = experiment.to_pandas()
    cols = [c for c in df.columns if c.startswith(("inputs.", "feedback."))]
    display(df[cols])
except Exception as e:
    print("Open the experiment in LangSmith to see the scores:", e)

Created dataset 'MADAR - Scenario Test Set' with 7 examples
View the evaluation results for experiment: 'MADAR-eval-fe7e04b5' at:
https://smith.langchain.com/o/7beab74e-4d97-4cae-8d6e-074d42b0ee6a/datasets/894e79b2-6058-42c3-ac7e-4240f923b332/compare?selectedSessions=f01bc5aa-39ca-4771-9db5-ca9bb5eefa44




0it [00:00, ?it/s]

,inputs.scenario,inputs.sat1_battery,inputs.sat2_battery,feedback.completion_rate_match,feedback.correct_satellite_selected,feedback.reassignment_correct,feedback.conflict_detection_correct,feedback.contract_net_protocol,feedback.scenario_verdict_match,inputs.conflict_missions
0,Communication Failure,50.0,95.0,1,1,1,1,1,1,NaN
1,Low Battery,NaN,20.0,1,1,1,1,1,1,NaN
2,Scheduling Conflict,NaN,NaN,1,1,1,1,1,1,5.0
3,Scheduling Conflict,NaN,NaN,1,1,1,1,1,1,3.0
4,Communication Failure,NaN,NaN,1,1,1,1,1,1,NaN
5,Normal Mission Assignment,NaN,NaN,1,1,1,1,1,1,NaN
6,Low Battery,NaN,NaN,1,1,1,1,1,1,NaN


In [19]:
import time
import gradio as gr
import pandas as pd


def _madar_ui_outputs(frame, result, is_final):
    m = frame["metrics"]
    kpis = (
        f"### Simulation Step: {frame['step']}  |  {frame['stage']}\n\n"
        f"**Mission Completion Rate:** {m['mission_completion_rate_pct']}% "
        f"({m['completed_missions']}/{m['total_missions']}) &nbsp;&nbsp; "
        f"**Scheduling Conflicts:** {m['scheduling_conflicts']} &nbsp;&nbsp; "
        f"**Successful Reassignments:** {m['successful_mission_reassignments']}"
    )
    if is_final:
        verdict = (
            f"### {'✅ PASS' if result['passed'] else '⚠️ FAIL / Check Settings'} "
            f"— {result['scenario']}\n\n"
            f"Initial owner of M001: **{result['initial_owner'] or 'None'}**. "
            f"Replacement: **{result['replacement_owner'] or 'Not applicable'}**."
        )
    else:
        verdict = "**Running simulation:** step-by-step status playback from this test run."

    agents = pd.DataFrame(frame["agent_rows"], columns=["Agent", "ID", "Status", "Details"])
    missions = pd.DataFrame(frame["mission_rows"], columns=["Mission", "Priority", "Status", "Satellite"])
    messages = pd.DataFrame(
        frame["message_rows"],
        columns=["ID", "From", "To", "Performative", "Content"],
    )
    events = "\n".join(frame["event_log"][-25:])
    summary = (
        {
            "scenario": result["scenario"],
            "test_passed": result["passed"],
            "inputs": result["inputs"],
            "metrics": result["metrics"],
            "final_mission_statuses": result["final_mission_statuses"],
        }
        if is_final else {}
    )
    return kpis, verdict, agents, missions, messages, events, summary


def madar_ui_run(scenario, sat1_battery, sat2_battery, threshold, mission_count):
    # One call = one independent traced simulation (no state leakage).
    result = run_madar_scenario(
        scenario=scenario,
        sat1_battery=sat1_battery,
        sat2_battery=sat2_battery,
        battery_threshold=threshold,
        conflict_missions=mission_count,
    )
    # Stream snapshots to the page so the user can see progression rather than just a final table.
    history = result["history"]
    for index, frame in enumerate(history):
        yield _madar_ui_outputs(frame, result, is_final=index == len(history) - 1)
        time.sleep(0.10)  # Visual playback of discrete simulation steps.


with gr.Blocks(title="MADAR | Satellite Mission Coordination") as madar_demo:
    gr.Markdown("# 🛰️ MADAR — Satellite Mission Coordination")
    gr.Markdown(
        "**Phase 2: Testing & User Interface.** "
        "Select a scenario, adjust satellite conditions, and run the simulation. "
        "The dashboard replays the captured simulation steps. "
        "Each run starts with a fresh environment."
    )

    with gr.Row():
        with gr.Column(scale=1):
            scenario_input = gr.Dropdown(
                choices=list(SCENARIOS), value=SCENARIOS[0],
                label="Testing Scenario"
            )
            battery1_input = gr.Slider(
                minimum=0, maximum=100, step=5, value=90, label="SAT-1 Battery (%)"
            )
            battery2_input = gr.Slider(
                minimum=0, maximum=100, step=5, value=80, label="SAT-2 Battery (%)"
            )
            threshold_input = gr.Slider(
                minimum=30, maximum=80, step=5, value=30, label="Battery Threshold (%)"
            )
            mission_count_input = gr.Slider(
                minimum=3, maximum=5, step=1, value=3,
                label="Number of Missions (Scheduling Conflict only)"
            )
            gr.Markdown(
                "**Default demonstration:** batteries 90% / 80%, threshold 30%. "
                "For Low Battery and Communication Failure, the fault is automatically "
                "injected into the initially assigned satellite. "
                "At least two eligible satellites are needed for successful reassignment."
            )
            run_button = gr.Button("▶ Run Simulation", variant="primary")
        with gr.Column(scale=2):
            kpi_output = gr.Markdown("### Run a scenario to view performance metrics")
            verdict_output = gr.Markdown("**No test run yet.**")
            agent_output = gr.Dataframe(label="Live Agent / Satellite Status", interactive=False)
            mission_output = gr.Dataframe(label="Mission Status", interactive=False)

    with gr.Row():
        message_output = gr.Dataframe(
            label="ACL-like Message Log (CFP, PROPOSE, ACCEPT_PROPOSAL, REJECT_PROPOSAL, INFORM, etc.)",
            interactive=False,
        )
    event_output = gr.Textbox(label="Environment Event Log", lines=9, interactive=False)
    summary_output = gr.JSON(label="Final Test Summary (for screenshots)")

    run_button.click(
        fn=madar_ui_run,
        inputs=[scenario_input, battery1_input, battery2_input,
                threshold_input, mission_count_input],
        outputs=[kpi_output, verdict_output, agent_output, mission_output,
                 message_output, event_output, summary_output],
        show_progress="minimal",
    )

madar_demo.queue()
madar_demo.launch(share=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

# **MADAR Dashboard (HTML) connected to LangSmith**


In [20]:
import os, json, glob, time, threading
from copy import deepcopy
from langsmith import traceable


def _ui_frame(stage, env, ma, gs, ca, co):
    """One snapshot in the exact format the HTML dashboard replays."""
    env.event_log.append(f"[Step {env.simulation_step}] {stage}")
    return deepcopy({
        "step": env.simulation_step,
        "stage": stage,
        "sats": [{
            "id": sid,
            "bat": s.battery_level,
            "st": s.satellite_status,
            "link": env.communication_status[sid],
            "m": s.assigned_mission["mission_id"] if s.assigned_mission else None,
            "plan": list(s.mission_plan),
        } for sid, s in env.satellites.items()],
        "ms": [{
            "id": mid, "loc": m["target_location"], "pri": m["priority"],
            "win": m["time_window"], "status": m["mission_status"],
            "sat": co._getAssignedSatelliteID(mid),
        } for mid, m in ma.missions.items()],
        "mc": len(ca.getMessageLog()),
        "ev": env.event_log[-40:],
        "conf": len(co.scheduling_conflicts),
        "re": len(set(co.successful_reassignments)),
        "done": sum(m["mission_status"] == "Completed" for m in ma.missions.values()),
        "total": len(ma.missions),
        "gs": len(gs.received_updates),
    })


@traceable(name="MADAR - Dashboard Run", run_type="chain")
def run_madar_dashboard(b1=90, b2=80, th=30, d1=0, d2=0, l1=0, l2=0, pri=("High",)):
    """Same inputs as the HTML page. Every agent call inside is traced by LangSmith."""
    b1, b2, th, d1, d2, l1, l2 = (int(x) for x in (b1, b2, th, d1, d2, l1, l2))
    pri = list(pri)
    if not (0 <= b1 <= 100 and 0 <= b2 <= 100):
        raise ValueError("Batteries must be 0..100")
    if not (30 <= th <= 80):
        raise ValueError("Threshold must be 30..80")
    if not (0 <= d1 <= 25 and 0 <= d2 <= 25):
        raise ValueError("Drain must be 0..25")
    if not (1 <= len(pri) <= 5) or any(p not in ("High", "Medium", "Low") for p in pri):
        raise ValueError("Use 1..5 missions with High / Medium / Low priority")

    env, ma, ba, gs, ca, co = _create_phase4_system(b1, b2, th)
    drain = {"SAT-1": d1, "SAT-2": d2}
    link_drop = {"SAT-1": l1, "SAT-2": l2}   # 0 stable, -1 off from start, N drops at step N
    frames = []
    cap = lambda stage: frames.append(_ui_frame(stage, env, ma, gs, ca, co))
    pending = lambda: any(m["mission_status"] == "Pending" for m in ma.missions.values())

    for i, p in enumerate(pri, start=1):
        _add_phase4_mission(env, ma, co, f"M{i:03}", p)
    cap("Mission requests received")

    for sid, drop in link_drop.items():
        if drop < 0:
            ca.updateConnectionStatus(sid, "Disconnected")
    co.processAlerts()
    co.assignPendingMissions()
    cap("Initial checks and allocation completed")

    def tick():
        for sid, sat in env.satellites.items():
            if sat.assigned_mission and drain[sid]:
                sat.battery_level = max(0, sat.battery_level - drain[sid])
            drop = link_drop[sid]
            if drop > 0 and env.simulation_step >= drop and env.communication_status[sid] == "Connected":
                ca.updateConnectionStatus(sid, "Disconnected")
        co.processAlerts()   # Battery Agent + Communication Agent -> Coordinator reassignment

    for _ in range(100):
        active = [s for s in env.satellites.values() if s.assigned_mission]
        if not active and not pending():
            break
        if not active:
            co.assignPendingMissions()
            active = [s for s in env.satellites.values() if s.assigned_mission]
            if not active:
                cap("No eligible satellite: remaining missions stay pending")
                break
        for sat in active:
            if not sat.assigned_mission:
                continue
            mid = sat.assigned_mission["mission_id"]
            if ma.getMission(mid)["mission_status"] != "In Progress":
                ma.updateMissionStatus(mid, "In Progress")
                co.assignment_status[mid] = "In Progress"
                co._notifyGroundStation(mid, sat.satellite_id, "In Progress", "Mission execution started")
            action = sat.executeMission()
            env.advanceSimulation()
            env.event_log.append(f"{sat.satellite_id} | {mid} | {action}")
            if sat.assigned_mission is None and action == f"Mission {mid} completed":
                ma.updateMissionStatus(mid, "Completed")
                co.assignment_status[mid] = "Completed"
                co._notifyGroundStation(mid, sat.satellite_id, "Completed", "Mission executed successfully")
            tick()
            cap(f"{sat.satellite_id}: {action}")
        if pending() and any(s.availability for s in env.satellites.values()):
            co.assignPendingMissions()
            cap("Attempted allocation of waiting missions")

    log = ca.getMessageLog()
    low = sorted({m["content"].get("satellite_id") for m in log
                  if m["initiator"] == "BatteryAgent"})
    lost = sorted({m["content"].get("satellite_id") for m in log
                   if m["initiator"] == "CommunicationAgent"})
    total = len(ma.missions)
    done = sum(m["mission_status"] == "Completed" for m in ma.missions.values())
    notes = []
    if low: notes.append("Low battery: " + ", ".join(low))
    if lost: notes.append("Link lost: " + ", ".join(lost))
    if co.scheduling_conflicts:
        notes.append("Scheduling conflict: " + ", ".join(dict.fromkeys(c["mission_id"] for c in co.scheduling_conflicts)))
    if co.successful_reassignments:
        notes.append("Reassigned: " + ", ".join(dict.fromkeys(co.successful_reassignments)))
    if total - done: notes.append(f"{total - done} mission(s) without an eligible satellite")
    if not notes: notes.append("No faults detected")
    cap(f"FINAL RESULT: {done}/{total} missions completed")

    return {
        "summary": {"inputs": {"b1": b1, "b2": b2, "th": th, "d1": d1, "d2": d2,
                               "l1": l1, "l2": l2, "pri": pri},
                    "metrics": _phase4_metrics(env, ma, co), "notes": notes},
        "final": {"all": done == total, "left": total - done, "notes": notes},
        "msgs": [{"id": m["message_id"], "f": m["initiator"], "t": m["responder"],
                  "p": m["performative"],
                  "c": json.dumps(m["content"], ensure_ascii=False, default=str)} for m in log],
        "frames": frames,
    }

# SERVER (serves the page + the API on the same address)
import uvicorn
from fastapi import FastAPI
from fastapi.responses import HTMLResponse, JSONResponse
from pydantic import BaseModel

# Upload MADAR.html to Colab
html_files = sorted(glob.glob("/content/MADAR*.html") + glob.glob("MADAR*.html"))
if not html_files:
    from google.colab import files
    print("Choose MADAR.html from your computer:")
    html_files = list(files.upload().keys())
HTML_PATH = html_files[0]
print("Dashboard file:", HTML_PATH)


class RunRequest(BaseModel):
    b1: int = 90
    b2: int = 80
    th: int = 30
    d1: int = 0
    d2: int = 0
    l1: int = 0
    l2: int = 0
    pri: list[str] = ["High"]


madar_api = FastAPI()


@madar_api.get("/", response_class=HTMLResponse)
def madar_page():
    with open(HTML_PATH, encoding="utf-8") as f:
        return f.read()


@madar_api.get("/api/health")
def madar_health():
    return {
        "backend": "python",
        "tracing": os.environ.get("LANGSMITH_TRACING") == "true"
                   and bool(os.environ.get("LANGSMITH_API_KEY")),
        "project": os.environ.get("LANGSMITH_PROJECT", "default"),
    }


@madar_api.post("/api/run")
def madar_run(req: RunRequest):
    try:
        return run_madar_dashboard(**req.model_dump())
    except ValueError as e:
        return JSONResponse({"error": str(e)}, status_code=400)


# Stop an older server if this cell is re-run
if "madar_server" in globals():
    madar_server.should_exit = True
    time.sleep(1.5)

PORT = 8050
madar_server = uvicorn.Server(uvicorn.Config(madar_api, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=madar_server.run, daemon=True).start()
time.sleep(2)

print("Tracing:", os.environ.get("LANGSMITH_TRACING"), "| Project:", os.environ.get("LANGSMITH_PROJECT"))
from google.colab import output
output.serve_kernel_port_as_window(PORT)   # click the link to open MADAR in a new tab

Choose MADAR.html from your computer:


Saving MADAR (1).html to MADAR (1).html
Dashboard file: MADAR (1).html
Tracing: true | Project: MADAR_Project
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>